# Stage 1 · 任务学习
复用已训练 OpenJev-4B，BF16 骨干 + LoRA，FP32 原决策头与交互层。先实测单卡 pilot，再开始正式训练。

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Reuse the prepared data; RUN_FULL_STAGE1 below controls training size.
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full


In [2]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    # BF16 + LoRA does not use torchao quantization. Colab's old optional
    # torchao package makes PEFT 0.19.1 fail even on ordinary Linear layers.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [3]:
%pip uninstall -y torchao

## 40GB A100：优化执行路径测试
保留 MODE='full' 读取已准备的数据，不重跑 00。默认仅 8 步；不会覆盖旧的 probe 或正式模型。
先运行下面的真实 Qwen3.5 小结构 FP32/BF16 损失、梯度检查，再做完整数据中大 K／长样本的无参数更新压力测试。
优化只改变执行：相同请求在同次前向中共享梯度图、分支按长度组批、跳过未参与损失的输出。候选归一化、精度、数据和损失权重不变。
新版单元格的旧输出已清空；旧 notebook 原件备份在本地 outputs/notebook_backups。

In [ ]:
# Small Qwen3.5 + LoRA check: no downloads, no API, no 4B training.
import torch
assert torch.cuda.is_available(), 'Select an A100 GPU runtime'
assert torch.cuda.is_bf16_supported(including_emulation=False), 'Native BF16 GPU required'
for check_dtype in ('float32', 'bfloat16'):
    log_path = ROOT / f'qwen_execution_{check_dtype}.log'
    cmd = [sys.executable, '-u', 'scripts/check_qwen_lora.py', '--device', 'cuda',
           '--compare-execution', '--dtype', check_dtype,
           '--output', str(ROOT/f'qwen_execution_{check_dtype}.json')]
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True); log.write(line); log.flush()
                status = process.wait()
            except KeyboardInterrupt:
                process.terminate(); process.wait(); raise
    if status:
        raise RuntimeError(f'Execution equivalence check failed; inspect {log_path}. Do not start 4B training.')


## 设置（在这里改）
当前用 a100_40gb；换到 80GB 后改 PROFILE，而不是改变数据 MODE。CHECKPOINT_MODE 默认 branch；若压力测试显存不足，可单独测试 branch_layer。所有配置都是待实测起点，不能保证最坏输入的显存。

In [ ]:
import copy, importlib, torch
from sa_jev import execution, model, trajectories, training, performance, experiments
for module in (execution, model, trajectories, training, performance, experiments):
    importlib.reload(module)
RUN_FULL_STAGE1 = False
PROBE_STEPS = 8
TRAIN_SEEDS = [17]
PROFILE = 'a100_40gb'  # Later: 'a100_80gb' or 'h100_80gb'.
CHECKPOINT_MODE = 'branch'  # Optional memory tradeoff: 'branch_layer'; compare measured speed.
STRESS_CHECK = True
config = performance.execution_config(config, PROFILE, stage=1, checkpoint_mode=CHECKPOINT_MODE)
config['training'].update(accumulation=8, validate_every=500, save_every=25)
print('ROOT:', ROOT, 'GPU:', torch.cuda.get_device_name(0), flush=True)
print('Formal training' if RUN_FULL_STAGE1 else f'Isolated {PROBE_STEPS}-step test; same effective batch of 8', flush=True)


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
from sa_jev.io import read_json
stage1_config = copy.deepcopy(config)
if not RUN_FULL_STAGE1:
    stage1_config['training'].update(validate_every=PROBE_STEPS, save_every=PROBE_STEPS)
probe_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:6]
for seed in (TRAIN_SEEDS if RUN_FULL_STAGE1 else [17]):
    probe_output = None if RUN_FULL_STAGE1 else ROOT/'runs/stage1_probe'/probe_id/f'seed_{seed}'
    history = experiments.train_variant(
        ROOT, stage1_config, 'stage1', seed,
        max_steps=None if RUN_FULL_STAGE1 else PROBE_STEPS,
        validation_limit=None if RUN_FULL_STAGE1 else 6,
        output_directory=probe_output, stress_check=STRESS_CHECK and not RUN_FULL_STAGE1,
    )
    output = probe_output or ROOT/'runs/stage1'/f'seed_{seed}'
    manifest = read_json(ROOT/'data/task_manifest.json')
    timing = performance.throughput_report(history, output,
        training_states=manifest['task_states'], effective_batch=8,
        epochs=config['training']['epochs'], validation_states=manifest['validation_states'],
        validate_every=config['training']['validate_every'], stage=1)
    print('Results/checkpoint:', output, flush=True)


## 看哪些结果
检查小模型 execution_comparison.passed、stress_probe.json、training.jsonl、throughput.json。
每步记录 task_seconds、dpo_seconds、实际分支数与补齐 token 数，loss、梯度范数、显存及 ETA。
8 步只是起点；要估算正式时间，使用同一 PROFILE 再测 20～30 步并核对长样本。不要把小模型耗时当作 4B 吞吐。
OOM 后重启会话，降低 branch_microbatch 或 branch_token_budget；不要截断证据或降低有效批量。branch_token_budget 只限制并行分组，超过它的单条长分支完整保留。
一天目标：两阶段各 36,000 状态、有效批量 8，共 9,000 次更新；两阶段平均单步之和须小于 19.2 秒，且还要留参考、验证、保存时间。未测得该吞吐前，不承诺一天内完成。